In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 800
ambulance_types = ['BASIC', 'ADVANCED', 'ICU']
emergency_types = ['Accident', 'Cardiac', 'Fire', 'Fall', 'Breathing', 'Assault', 'Other']
severities = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

data = pd.DataFrame({
    'emergency_type': np.random.choice(emergency_types, n),
    'severity': np.random.choice(severities, n, p=[0.3, 0.35, 0.25, 0.10]),
    'ambulance_type': np.random.choice(ambulance_types, n, p=[0.5, 0.3, 0.2]),
    'distance_km': np.round(np.random.uniform(0.2, 25, n), 2),
    'ambulance_available': np.random.choice([1, 0], n, p=[0.7, 0.3]),
})

# What type of ambulance a severity level ideally needs
required_type = data['severity'].map({'LOW': 'BASIC', 'MEDIUM': 'BASIC', 'HIGH': 'ADVANCED', 'CRITICAL': 'ICU'})
type_match_bonus = (data['ambulance_type'] == required_type).astype(float) * 0.3

# Suitability score: closer is better, matching type is better, must be available
distance_penalty = data['distance_km'] / 25  # 0 (close) to 1 (far)
availability_penalty = (1 - data['ambulance_available']) * 0.5

data['suitability_score'] = np.clip(
    1.0 - distance_penalty + type_match_bonus - availability_penalty + np.random.normal(0, 0.03, n),
    0, 1.3
)

data.head(10)

,emergency_type,severity,ambulance_type,distance_km,ambulance_available,suitability_score
0,Other,HIGH,BASIC,2.98,1,0.889953
1,Fall,HIGH,ICU,9.03,1,0.611512
2,Breathing,LOW,BASIC,24.46,0,0.000000
3,Other,MEDIUM,BASIC,19.89,1,0.490788
4,Fire,MEDIUM,ADVANCED,12.53,1,0.518277
5,Breathing,LOW,BASIC,8.63,1,0.977522
6,Breathing,MEDIUM,ICU,21.83,0,0.000000
7,Other,MEDIUM,ADVANCED,13.71,1,0.417894
8,Cardiac,HIGH,ADVANCED,19.01,1,0.585557
9,Fire,CRITICAL,ADVANCED,18.90,0,0.000000


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = data[['emergency_type', 'severity', 'ambulance_type', 'distance_km', 'ambulance_available']]
y = data['suitability_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['emergency_type', 'severity', 'ambulance_type']),
], remainder='passthrough')

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42)),
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error: {mae:.4f}")

Mean Absolute Error: 0.0549


In [3]:
import joblib

joblib.dump(model, '../trained_models/ambulance_recommendation_model.pkl')
print("Model saved successfully!")

# Test: a CRITICAL emergency, nearby ICU ambulance that's available
test_case = pd.DataFrame({
    'emergency_type': ['Cardiac'],
    'severity': ['CRITICAL'],
    'ambulance_type': ['ICU'],
    'distance_km': [3.0],
    'ambulance_available': [1],
})
print(f"Nearby available ICU ambulance for critical case: {model.predict(test_case)[0]:.3f}")

# Compare: same emergency, but a far-away unavailable BASIC ambulance
test_case_2 = pd.DataFrame({
    'emergency_type': ['Cardiac'],
    'severity': ['CRITICAL'],
    'ambulance_type': ['BASIC'],
    'distance_km': [20.0],
    'ambulance_available': [0],
})
print(f"Far unavailable BASIC ambulance for same case: {model.predict(test_case_2)[0]:.3f}")

Model saved successfully!
Nearby available ICU ambulance for critical case: 0.880
Far unavailable BASIC ambulance for same case: 0.004
